# Homework — Part 2: Training Neural Networks with Custom Modules

**Содержание:**
1. Оптимизаторы (SGD + momentum, Adam)
2. LR-scheduler (cosine, step) + warmup + early stopping
3. Регрессия (synthetic, multi-target, 3 размера модели, 5 активаций)
4. Классификация (load_digits CNN)
5. Автоэнкодер (свёрточный, Digits 8×8)

> **Датасеты:** синтетические / sklearn-встроенные (сеть недоступна, поэтому MNIST заменён на load_digits 8×8 — аналогичная задача).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erf
from sklearn.datasets import make_regression, load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)  # фиксируем seed для воспроизводимости, иначе каждый раз разные результаты

## Часть 1 — Слои, активации и критерии (из homework_modules)

In [ ]:
class Module:
    def __init__(self): self.output=None; self.gradInput=None; self.training=True
    def forward(self, x): return self.updateOutput(x)
    def backward(self, x, g):
        self.updateGradInput(x,g); self.accGradParameters(x,g); return self.gradInput
    def updateOutput(self, x): pass
    def updateGradInput(self, x, g): pass
    def accGradParameters(self, x, g): pass
    def zeroGradParameters(self): pass
    def getParameters(self): return []
    def getGradParameters(self): return []
    def train(self): self.training=True
    def evaluate(self): self.training=False

class Sequential(Module):
    def __init__(self): super().__init__(); self.modules=[]; self._ins=[]
    def add(self,m): self.modules.append(m); return self
    def updateOutput(self, x):
        self._ins=[x]; cur=x
        for m in self.modules: cur=m.forward(cur); self._ins.append(cur)
        self.output=cur; return cur
    def backward(self, x, g):
        for i in reversed(range(len(self.modules))):
            g=self.modules[i].backward(self._ins[i],g)
        self.gradInput=g; return g
    def zeroGradParameters(self):
        for m in self.modules: m.zeroGradParameters()
    def getParameters(self):   return [m.getParameters()   for m in self.modules]
    def getGradParameters(self): return [m.getGradParameters() for m in self.modules]
    def train(self):
        self.training=True
        for m in self.modules: m.train()
    def evaluate(self):
        self.training=False
        for m in self.modules: m.evaluate()

class Linear(Module):
    def __init__(self, ni, no):
        super().__init__(); s=1./np.sqrt(ni)
        self.W=np.random.uniform(-s,s,(no,ni)).astype(np.float32)
        self.b=np.random.uniform(-s,s,no).astype(np.float32)
        self.gradW=np.zeros_like(self.W); self.gradb=np.zeros_like(self.b)
    def updateOutput(self, x): self.output=x@self.W.T+self.b; return self.output
    def updateGradInput(self, x, g): self.gradInput=g@self.W; return self.gradInput
    def accGradParameters(self, x, g): self.gradW=g.T@x; self.gradb=g.sum(0)
    def zeroGradParameters(self): self.gradW.fill(0); self.gradb.fill(0)
    def getParameters(self): return [self.W, self.b]
    def getGradParameters(self): return [self.gradW, self.gradb]
    def __repr__(self): return f"Linear({self.W.shape[1]}→{self.W.shape[0]})"

class BatchNormalization(Module):
    EPS=1e-3
    def __init__(self, alpha=0.):
        super().__init__(); self.alpha=alpha; self.mu=None; self.var=None
    def updateOutput(self, x):
        if self.training:
            bm=x.mean(0); bv=x.var(0)
            if self.mu is None: self.mu=bm.copy(); self.var=bv.copy()
            else: self.mu=self.mu*self.alpha+bm*(1-self.alpha); self.var=self.var*self.alpha+bv*(1-self.alpha)
            self._bm=bm; self._bv=bv
            self.output=(x-bm)/np.sqrt(bv+self.EPS)
        else:
            self.output=(x-self.mu)/np.sqrt(self.var+self.EPS)
        return self.output
    def updateGradInput(self, x, g):
        N=x.shape[0]; std=np.sqrt(self._bv+self.EPS); xh=(x-self._bm)/std
        self.gradInput=(1/(N*std))*(N*g-g.sum(0)-xh*(g*xh).sum(0)); return self.gradInput

class ChannelwiseScaling(Module):
    def __init__(self, n):
        super().__init__(); s=1./np.sqrt(n)
        self.gamma=np.random.uniform(-s,s,n).astype(np.float32)
        self.beta=np.random.uniform(-s,s,n).astype(np.float32)
        self.dg=np.zeros(n,np.float32); self.db=np.zeros(n,np.float32)
    def updateOutput(self, x): self.output=x*self.gamma+self.beta; return self.output
    def updateGradInput(self, x, g): self.gradInput=g*self.gamma; return self.gradInput
    def accGradParameters(self, x, g): self.db=g.sum(0); self.dg=(g*x).sum(0)
    def zeroGradParameters(self): self.dg.fill(0); self.db.fill(0)
    def getParameters(self): return [self.gamma, self.beta]
    def getGradParameters(self): return [self.dg, self.db]

class Dropout(Module):
    def __init__(self, p=0.5): super().__init__(); self.p=p; self.mask=None
    def updateOutput(self, x):
        if self.training:
            self.mask=(np.random.rand(*x.shape)>self.p).astype(x.dtype)
            self.output=x*self.mask/(1-self.p)
        else: self.output=x.copy()
        return self.output
    def updateGradInput(self, x, g):
        self.gradInput=g*self.mask/(1-self.p) if self.training else g.copy(); return self.gradInput

class ReLU(Module):
    def updateOutput(self, x): self.output=np.maximum(x,0); return self.output
    def updateGradInput(self, x, g): self.gradInput=g*(x>0); return self.gradInput
    def __repr__(self): return "ReLU"

class LeakyReLU(Module):
    def __init__(self, s=0.03): super().__init__(); self.s=s
    def updateOutput(self, x): self.output=np.where(x>=0,x,self.s*x); return self.output
    def updateGradInput(self, x, g): self.gradInput=g*np.where(x>=0,1.,self.s); return self.gradInput

class ELU(Module):
    def __init__(self, a=1.0): super().__init__(); self.a=a
    def updateOutput(self, x): self.output=np.where(x>=0,x,self.a*(np.exp(x)-1)); return self.output
    def updateGradInput(self, x, g): self.gradInput=g*np.where(x>=0,1.,self.a*np.exp(x)); return self.gradInput

class Gelu(Module):
    def updateOutput(self, x):
        self.output=x*0.5*(1+erf(x/np.sqrt(2))); return self.output
    def updateGradInput(self, x, g):
        P=0.5*(1+erf(x/np.sqrt(2))); ph=np.exp(-0.5*x**2)/np.sqrt(2*np.pi)
        self.gradInput=g*(P+x*ph); return self.gradInput

class SoftPlus(Module):
    def updateOutput(self, x):
        self.output=np.log1p(np.exp(-np.abs(x)))+np.maximum(x,0); return self.output
    def updateGradInput(self, x, g):
        self.gradInput=g/(1+np.exp(-x)); return self.gradInput

class Sigmoid(Module):
    def updateOutput(self, x):
        self.output=1/(1+np.exp(-np.clip(x,-30,30))); return self.output
    def updateGradInput(self, x, g):
        s=self.output; self.gradInput=g*s*(1-s); return self.gradInput

class LogSoftMax(Module):
    def updateOutput(self, x):
        s=x-x.max(1,keepdims=True); self.output=s-np.log(np.exp(s).sum(1,keepdims=True)); return self.output
    def updateGradInput(self, x, g):
        sm=np.exp(self.output); self.gradInput=g-sm*g.sum(1,keepdims=True); return self.gradInput

# ─────────────────────────────── CONV2D ──────────────────────────────────────
class Conv2d(Module):
    def __init__(self, ci, co, ks, stride=1, padding=0, bias=True, padding_mode='zeros'):
        super().__init__()
        self.ci=ci; self.co=co
        self.ks=(ks,ks) if isinstance(ks,int) else ks
        self.st=(stride,stride) if isinstance(stride,int) else stride
        self.pd=(padding,padding) if isinstance(padding,int) else padding
        self.use_bias=bias
        kH,kW=self.ks
        s=1./np.sqrt(ci*kH*kW)
        self.weight=np.random.uniform(-s,s,(co,ci,kH,kW)).astype(np.float32)
        self.bias_arr=np.zeros(co,np.float32) if bias else None
        self.dW=np.zeros_like(self.weight); self.db=np.zeros(co,np.float32) if bias else None

    # fast im2col via stride tricks
    def _im2col(self, xp, kH, kW, sH, sW, oH, oW):
        N,C,_,_=xp.shape
        shape=(N,oH,oW,C,kH,kW)
        st=(xp.strides[0],sH*xp.strides[2],sW*xp.strides[3],xp.strides[1],xp.strides[2],xp.strides[3])
        return np.lib.stride_tricks.as_strided(xp,shape,st).reshape(N,oH*oW,C*kH*kW)

    def _col2im(self, dc, N, C, H, W, kH, kW, sH, sW, pH, pW, oH, oW):
        dx=np.zeros((N,C,H+2*pH,W+2*pW),dtype=dc.dtype)
        for i in range(oH):
            for j in range(oW):
                dx[:,:,i*sH:i*sH+kH,j*sW:j*sW+kW]+=dc[:,:,i*oW+j].reshape(N,C,kH,kW)
        return dx

    def updateOutput(self, x):
        N,Ci,H,W=x.shape; kH,kW=self.ks; sH,sW=self.st; pH,pW=self.pd
        xp=np.pad(x,((0,0),(0,0),(pH,pH),(pW,pW))) if pH or pW else x
        self._xp=xp
        oH=(H+2*pH-kH)//sH+1; oW=(W+2*pW-kW)//sW+1
        col=self._im2col(xp,kH,kW,sH,sW,oH,oW)
        Wm=self.weight.reshape(self.co,-1)
        out=(col@Wm.T).transpose(0,2,1).reshape(N,self.co,oH,oW)
        if self.use_bias and self.bias_arr is not None: out=out+self.bias_arr[None,:,None,None]
        self.output=out; return out

    def updateGradInput(self, x, g):
        N,Ci,H,W=x.shape; kH,kW=self.ks; sH,sW=self.st; pH,pW=self.pd
        oH,oW=g.shape[2],g.shape[3]
        Wm=self.weight.reshape(self.co,-1)
        go=g.reshape(N,self.co,oH*oW)
        dc=np.tensordot(Wm.T,go,axes=([1],[1])).transpose(1,0,2)
        dxp=self._col2im(dc,N,Ci,H,W,kH,kW,sH,sW,pH,pW,oH,oW)
        self.gradInput=dxp[:,:,pH:pH+H,pW:pW+W] if pH or pW else dxp; return self.gradInput

    def accGradParameters(self, x, g):
        kH,kW=self.ks; sH,sW=self.st; pH,pW=self.pd; oH,oW=g.shape[2],g.shape[3]; N=x.shape[0]
        col=self._im2col(self._xp,kH,kW,sH,sW,oH,oW)
        go=g.reshape(N,self.co,oH*oW)
        # go: (N, co, oH*oW), col: (N, oH*oW, C*kH*kW)
        # dW[co, CkHkW] = sum_{n,hw} go[n,co,hw] * col[n,hw,CkHkW]
        self.dW = np.einsum('nch,nhk->ck', go, col).reshape(self.weight.shape)
        if self.use_bias and self.db is not None: self.db=g.sum((0,2,3))

    def zeroGradParameters(self):
        self.dW.fill(0)
        if self.db is not None: self.db.fill(0)
    def getParameters(self):
        return [self.weight, self.bias_arr] if self.use_bias and self.bias_arr is not None else [self.weight]
    def getGradParameters(self):
        return [self.dW, self.db] if self.use_bias and self.db is not None else [self.dW]

# ─────────────────────────────── POOLING ─────────────────────────────────────
class MaxPool2d(Module):
    def __init__(self, k, s, p=0): super().__init__(); self.k=k; self.s=s; self.p=p
    def updateOutput(self, x):
        N,C,H,W=x.shape; k,s,p=self.k,self.s,self.p
        xp=np.pad(x,((0,0),(0,0),(p,p),(p,p)),constant_values=-np.inf) if p else x
        oH=(H+2*p-k)//s+1; oW=(W+2*p-k)//s+1
        self.output=np.zeros((N,C,oH,oW),dtype=x.dtype)
        for i in range(oH):
            for j in range(oW):
                self.output[:,:,i,j]=xp[:,:,i*s:i*s+k,j*s:j*s+k].max((-2,-1))
        return self.output
    def updateGradInput(self, x, g):
        N,C,H,W=x.shape; k,s,p=self.k,self.s,self.p; oH,oW=g.shape[2],g.shape[3]
        xp=np.pad(x,((0,0),(0,0),(p,p),(p,p)),constant_values=-np.inf) if p else x
        dx=np.zeros_like(xp)
        for i in range(oH):
            for j in range(oW):
                pt=xp[:,:,i*s:i*s+k,j*s:j*s+k]; mv=pt.max((-2,-1),keepdims=True)
                mask=(pt==mv).astype(float); mask/=mask.sum((-2,-1),keepdims=True)
                dx[:,:,i*s:i*s+k,j*s:j*s+k]+=mask*g[:,:,i:i+1,j:j+1]
        self.gradInput=dx[:,:,p:p+H,p:p+W] if p else dx; return self.gradInput

class Flatten(Module):
    def __init__(self, s=0, e=-1): super().__init__(); self.s=s; self.e=e
    def updateOutput(self, x):
        nd=x.ndim; s=self.s%nd; e=self.e%nd
        self.output=x.reshape(x.shape[:s]+(-1,)+x.shape[e+1:]); return self.output
    def updateGradInput(self, x, g): self.gradInput=g.reshape(x.shape); return self.gradInput

class Upsample2d(Module):
    def __init__(self, sc=2): super().__init__(); self.sc=sc
    def updateOutput(self, x):
        self.output=np.repeat(np.repeat(x,self.sc,-1),self.sc,-2); return self.output
    def updateGradInput(self, x, g):
        N,C,H,W=x.shape; sc=self.sc
        self.gradInput=g.reshape(N,C,H,sc,W,sc).sum((3,5)); return self.gradInput

class Reshape(Module):
    def __init__(self, *sh): super().__init__(); self.sh=sh
    def updateOutput(self, x):
        self._in=x.shape; self.output=x.reshape((x.shape[0],)+self.sh); return self.output
    def updateGradInput(self, x, g): self.gradInput=g.reshape(self._in); return self.gradInput

# ─────────────────────────────── CRITERIONS ──────────────────────────────────
class Criterion:
    def __init__(self): self.output=None; self.gradInput=None
    def forward(self,i,t): return self.updateOutput(i,t)
    def backward(self,i,t): return self.updateGradInput(i,t)
    def updateOutput(self,i,t): return self.output
    def updateGradInput(self,i,t): return self.gradInput

class MSECriterion(Criterion):
    def updateOutput(self,i,t): self.output=float(np.sum((i-t)**2)/i.shape[0]); return self.output
    def updateGradInput(self,i,t): self.gradInput=2*(i-t)/i.shape[0]; return self.gradInput

class MSE4D(Criterion):
    """MSE для произвольных тензоров (используется в автоэнкодере)."""
    def updateOutput(self,i,t): self.output=float(np.mean((i-t)**2)); return self.output
    def updateGradInput(self,i,t):
        self.gradInput=2*(i-t)/(i.size/i.shape[0])/i.shape[0]; return self.gradInput

class MAECriterion(Criterion):
    def updateOutput(self,i,t): self.output=float(np.mean(np.abs(i-t))); return self.output
    def updateGradInput(self,i,t): self.gradInput=np.sign(i-t)/i.shape[0]; return self.gradInput

class ClassNLLCriterion(Criterion):
    def updateOutput(self,i,t): self.output=-float(np.sum(t*i)/i.shape[0]); return self.output
    def updateGradInput(self,i,t): self.gradInput=-t/i.shape[0]; return self.gradInput

# ─── Extra modules (Reshape, Upsample2d, Sigmoid) ───────────────────────────

class Reshape(Module):
    """Reshape (N, flat) → (N, C, H, W) for decoder."""
    def __init__(self, *shape):
        super().__init__(); self.shape = shape
    def updateOutput(self, x):
        self._s = x.shape
        self.output = x.reshape((x.shape[0],) + self.shape)
        return self.output
    def updateGradInput(self, x, g):
        self.gradInput = g.reshape(self._s); return self.gradInput

class Upsample2d(Module):
    """Nearest-neighbour spatial upsampling by integer factor."""
    def __init__(self, factor):
        super().__init__(); self.f = factor
    def updateOutput(self, x):
        f = self.f
        self.output = np.repeat(np.repeat(x, f, axis=2), f, axis=3)
        return self.output
    def updateGradInput(self, x, g):
        f = self.f; N,C,H,W = x.shape
        self.gradInput = g.reshape(N,C,H,f,W,f).sum(axis=(3,5))
        return self.gradInput

class Sigmoid(Module):
    """Element-wise sigmoid, numerically stable."""
    def __init__(self): super().__init__()
    def updateOutput(self, x):
        self.output = np.where(x>=0, 1/(1+np.exp(-x)), np.exp(x)/(1+np.exp(x)))
        return self.output
    def updateGradInput(self, x, g):
        s = self.output; self.gradInput = g*s*(1-s); return self.gradInput

# ─── Extra criterions (MSE4D, MAECriterion) ──────────────────────────────────

class MSE4D(Criterion):
    """MSE over all elements — for 4-D autoencoder output."""
    def __init__(self): super().__init__()
    def updateOutput(self, p, t):
        self.output = float(np.mean((p-t)**2)); return self.output
    def updateGradInput(self, p, t):
        self.gradInput = 2*(p-t)/p.size; return self.gradInput

class MAECriterion(Criterion):
    """Mean Absolute Error loss."""
    def __init__(self): super().__init__()
    def updateOutput(self, p, t):
        self.output = float(np.mean(np.abs(p-t))); return self.output
    def updateGradInput(self, p, t):
        self.gradInput = np.sign(p-t)/p.size; return self.gradInput


## 1. Оптимизаторы

Реализованы **SGD с моментумом** и **Adam** — оба работают с любым `Sequential`-моделью через
`model.getParameters()` / `model.getGradParameters()`.
Поддерживают `set_lr()` для планировщиков.

In [ ]:
# OPTIMIZERS
def _flat(lst):
    out=[]
    for x in lst:
        if isinstance(x,list): out.extend(_flat(x))
        elif isinstance(x,np.ndarray): out.append(x)
    return out

class SGD:
    # SGD с моментумом — классика, работает хорошо если правильно подобрать lr
    def __init__(self, m, lr=0.01, momentum=0.9, wd=0.):
        self.m=m; self.lr=lr; self.mu=momentum; self.wd=wd; self._v=None
    def step(self):
        P,G=_flat(self.m.getParameters()),_flat(self.m.getGradParameters())
        if self._v is None: self._v=[np.zeros_like(p) for p in P]
        for p,g,v in zip(P,G,self._v):
            # v = momentum*v + lr*grad, потом param -= v
            v*=self.mu; v+=self.lr*(g+self.wd*p if self.wd else g); p-=v
    def zero_grad(self): self.m.zeroGradParameters()
    def set_lr(self, lr): self.lr=lr

class Adam:
    # Adam адаптирует lr для каждого параметра — обычно лучше SGD по умолчанию
    # TODO: добавить AMSGrad вариант когда-нибудь
    def __init__(self, m, lr=1e-3, betas=(0.9,0.999), eps=1e-8, wd=0.):
        self.m=m; self.lr=lr; self.b1,self.b2=betas; self.eps=eps; self.wd=wd
        self._mm=None; self._v=None; self._t=0
    def step(self):
        P,G=_flat(self.m.getParameters()),_flat(self.m.getGradParameters())
        if self._mm is None: self._mm=[np.zeros_like(p) for p in P]; self._v=[np.zeros_like(p) for p in P]
        self._t+=1; lt=self.lr*np.sqrt(1-self.b2**self._t)/(1-self.b1**self._t)  # bias correction
        for p,g,m,v in zip(P,G,self._mm,self._v):
            ge=g+self.wd*p if self.wd else g
            m*=self.b1; m+=(1-self.b1)*ge; v*=self.b2; v+=(1-self.b2)*ge*ge
            p-=lt*m/(np.sqrt(v)+self.eps)
    def zero_grad(self): self.m.zeroGradParameters()
    def set_lr(self, lr): self.lr=lr

## 2. LR-Scheduler, Warmup, Early Stopping, Trainer

`Trainer` инкапсулирует весь обучающий цикл:
- **Warmup** — линейный разогрев LR в первые `warmup` эпох  
- **Scheduler** — cosine annealing (или step decay) после разогрева  
- **Early stopping** — остановка при отсутствии улучшения за `patience` эпох  
- **Сохранение лучшей модели** — восстанавливает лучшие веса после обучения  
- **Switch**: принимает любой критерий, оптимизатор и метрику как параметры

In [ ]:
# LR SCHEDULERS
def cosine_lr(base, ep, total, eta_min=1e-6):
    return eta_min+0.5*(base-eta_min)*(1+np.cos(np.pi*ep/max(total,1)))

def step_lr(base, ep, step=5, gamma=0.5):
    return base*(gamma**(ep//step))

def r2_score(pred, target):
    # R² = 1 - SS_res/SS_tot, показывает долю объяснённой дисперсии
    # при R²<0 модель хуже чем просто предсказывать среднее — плохой знак
    ss_res = np.sum((target - pred) ** 2)
    ss_tot = np.sum((target - target.mean(axis=0, keepdims=True)) ** 2)
    return float(1.0 - ss_res / max(ss_tot, 1e-12))

def save_checkpoint(path, model):
    # сохраняем веса в .npz — numpy-архив, читается быстро
    import pathlib
    pathlib.Path(path).parent.mkdir(parents=True, exist_ok=True)
    flat = []
    def rec(p):
        if isinstance(p, list):
            for x in p: rec(x)
        else:
            flat.append(np.copy(p))
    rec(model.getParameters())
    np.savez_compressed(str(path), *flat)

def load_checkpoint(path, model):
    # восстанавливаем веса из файла — порядок должен совпадать
    z = np.load(path)
    flat = [z[k] for k in sorted(z.files)]
    it = iter(flat)
    def rec(p):
        if isinstance(p, list):
            for x in p: rec(x)
        else:
            p[...] = next(it)
    rec(model.getParameters())

class Trainer:
    def __init__(self, model, crit, opt, metric_fn=None, sched=None,
                 checkpoint_path=None):
        self.model=model; self.crit=crit; self.opt=opt
        self.mfn=metric_fn; self.sched=sched
        self.checkpoint_path=checkpoint_path
        self.tl=[]; self.vl=[]; self.tm=[]; self.vm=[]
        self.best=np.inf; self._bw=None; self._pat=0  # pat = счётчик плохих эпох

    def _run(self, X, y, bs, train):
        n=X.shape[0]; idx=np.random.permutation(n) if train else np.arange(n)
        ls=[]; ms=[]
        for s in range(0,n,bs):
            bi=idx[s:s+bs]; xb,yb=X[bi],y[bi]
            p=self.model.forward(xb)
            ls.append(float(self.crit.forward(p,yb)))
            if self.mfn: ms.append(float(self.mfn(p,yb)))
            if train:
                self.opt.zero_grad()
                self.model.backward(xb,self.crit.backward(p,yb))
                self.opt.step()
        return np.mean(ls),(np.mean(ms) if ms else None)

    def fit(self, Xt,yt,Xv,yv, epochs=50,bs=64,patience=10,warmup=3,
            lr=None, min_delta=0.0, verbose=True):
        lr=lr or self.opt.lr
        for ep in range(1,epochs+1):
            # warmup: линейно растём до lr, потом scheduler
            if ep<=warmup: cur=lr*ep/warmup
            elif self.sched: cur=self.sched(lr,ep-warmup,epochs-warmup)
            else: cur=lr
            self.opt.set_lr(cur)
            self.model.train(); tl,tm=self._run(Xt,yt,bs,True)
            self.model.evaluate(); vl,vm=self._run(Xv,yv,bs,False)
            self.tl.append(tl); self.vl.append(vl)
            if tm is not None: self.tm.append(tm); self.vm.append(vm)
            # сохраняем если val loss улучшился больше чем на min_delta
            if vl < self.best - min_delta:
                self.best=vl
                self._bw=[p.copy() for p in _flat(self.model.getParameters())]
                if self.checkpoint_path:
                    save_checkpoint(self.checkpoint_path, self.model)
                self._pat=0
            else: self._pat+=1
            if verbose and ep%max(1,epochs//8)==0:
                s=f"  ep{ep:3}/{epochs} lr={cur:.1e} | tr={tl:.4f} vl={vl:.4f}"
                if vm is not None: s+=f" | metric={vm:.4f}"
                print(s)
            if self._pat>=patience:
                verbose and print(f"  ↳ early stop @ ep{ep}"); break
        # Восстанавливаем лучшие веса
        if self._bw:
            for p,w in zip(_flat(self.model.getParameters()),self._bw): p[:]=w

    def plot(self, title="", mname="Metric", path=None):
        has=bool(self.tm); fig,axs=plt.subplots(1,2 if has else 1,figsize=(12,4))
        if not has: axs=[axs]
        axs[0].plot(self.tl,'b',label='train'); axs[0].plot(self.vl,'r',label='val')
        axs[0].set(title=f'{title} Loss',xlabel='Epoch'); axs[0].legend(); axs[0].grid()
        if has:
            axs[1].plot(self.tm,'b',label='train'); axs[1].plot(self.vm,'r',label='val')
            axs[1].set(title=f'{title} {mname}',xlabel='Epoch'); axs[1].legend(); axs[1].grid()
        plt.tight_layout()
        if path: plt.savefig(path,dpi=90)
        plt.show()


## 3. Мультирегрессия (3 целевых переменных, 15 фичей, 8000 сэмплов)

**Датасет:** `sklearn.make_regression` — синтетический, 10 информативных признаков из 15, шум σ=15.  
**Лосс:** MSECriterion (+ сравнение с MAE)  
**Метрика:** RMSE на тест-сете  
**Архитектура:** FCNN + BatchNorm + ChannelwiseScaling + Dropout

In [ ]:
# A.
print("\n═══  A. REGRESSION  ═══")
import pathlib
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

CKPT_DIR = pathlib.Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)  # папка для чекпоинтов

X,Y=make_regression(8000,15,n_informative=10,n_targets=3,noise=15.,random_state=42)
X,Y=X.astype(np.float32),Y.astype(np.float32)
Xt,Xv,Yt,Yv=train_test_split(X,Y,test_size=0.2,random_state=0)
Xv,Xte,Yv,Yte=train_test_split(Xv,Yv,test_size=0.5,random_state=0)
sx,sy=StandardScaler(),StandardScaler()
Xt=sx.fit_transform(Xt).astype(np.float32); Xv=sx.transform(Xv).astype(np.float32); Xte=sx.transform(Xte).astype(np.float32)
Yt=sy.fit_transform(Yt).astype(np.float32); Yv=sy.transform(Yv).astype(np.float32); Yte=sy.transform(Yte).astype(np.float32)

def rmse(p,t): return float(np.sqrt(np.mean((p-t)**2)))

def build_reg(hs, Act=ReLU):
    m=Sequential(); d=15
    for h in hs:
        m.add(Linear(d,h)).add(BatchNormalization(0.9)).add(ChannelwiseScaling(h))
        m.add(Act()).add(Dropout(0.2)); d=h
    return m.add(Linear(d,3))

# три размера: смотрим есть ли overfitting на big и underfitting на small
trainers_sz = {}
for name,hs in [('big',[256,128,64]),('medium',[64,32]),('small',[16])]:
    np.random.seed(42)
    mdl=build_reg(hs)
    tr=Trainer(mdl, MSECriterion(), Adam(mdl,lr=1e-3), rmse, cosine_lr,
               checkpoint_path=CKPT_DIR/f"reg_{name}.npz")
    tr.fit(Xt,Yt,Xv,Yv,epochs=40,bs=128,patience=8,warmup=3,lr=1e-3,verbose=False)
    mdl.evaluate()
    pred_te = mdl.forward(Xte)
    r = rmse(pred_te, Yte)
    r2 = r2_score(pred_te, Yte)
    print(f"  {name:8s} | test RMSE={r:.4f}  R²={r2:.4f}  ckpt=reg_{name}.npz")
    if r2 < 0.5:
        print(f"    !! R² низкий для {name} — возможно underfitting или плохие гиперпараметры")
    trainers_sz[name] = tr

# разные активации на medium-архитектуре — интересно насколько GELU лучше ReLU
print()
for an,Ac in [('ReLU',ReLU),('ELU',ELU),('Leaky',LeakyReLU),('Gelu',Gelu),('SoftPlus',SoftPlus)]:
    np.random.seed(42); mdl=build_reg([64,32],Ac)
    tr=Trainer(mdl,MSECriterion(),Adam(mdl,lr=1e-3),rmse,cosine_lr)
    tr.fit(Xt,Yt,Xv,Yv,epochs=35,bs=128,patience=7,warmup=3,lr=1e-3,verbose=False)
    mdl.evaluate()
    pred_te = mdl.forward(Xte)
    print(f"  Act={an:10s} | RMSE={rmse(pred_te,Yte):.4f}  R²={r2_score(pred_te,Yte):.4f}")

# сравниваем Adam vs SGD — обычно Adam быстрее сходится
print()
for on,of,kw in [('Adam',Adam,dict(lr=1e-3)),('SGD',SGD,dict(lr=0.05,momentum=0.9))]:
    np.random.seed(42); mdl=build_reg([64,32]); opt=of(mdl,**kw)
    tr=Trainer(mdl,MSECriterion(),opt,rmse,cosine_lr)
    tr.fit(Xt,Yt,Xv,Yv,epochs=35,bs=128,patience=8,warmup=3,lr=kw['lr'],verbose=False)
    mdl.evaluate()
    pred_te = mdl.forward(Xte)
    print(f"  Opt={on:5s} | RMSE={rmse(pred_te,Yte):.4f}  R²={r2_score(pred_te,Yte):.4f}")

# MAE более устойчив к выбросам чем MSE — посмотрим насколько разница заметна
print()
for ln,Cr in [('MSE',MSECriterion),('MAE',MAECriterion)]:
    np.random.seed(42); mdl=build_reg([64,32])
    tr=Trainer(mdl,Cr(),Adam(mdl,lr=1e-3),rmse,cosine_lr)
    tr.fit(Xt,Yt,Xv,Yv,epochs=35,bs=128,patience=7,warmup=3,lr=1e-3,verbose=False)
    mdl.evaluate()
    pred_te = mdl.forward(Xte)
    print(f"  Loss={ln:4s} | RMSE={rmse(pred_te,Yte):.4f}  R²={r2_score(pred_te,Yte):.4f}")


In [ ]:
# Кривые обучения — 3 размера регрессионной модели
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, name in zip(axes, ['big', 'medium', 'small']):
    tr = trainers_sz[name]
    ax.plot(tr.tl, 'b', label='train loss')
    ax.plot(tr.vl, 'r', label='val loss')
    ax.set_title(f'Регрессия — {name}')
    ax.set_xlabel('Epoch')
    ax.legend(); ax.grid(True)
plt.suptitle('Кривые обучения — регрессия')
plt.tight_layout(); plt.show()


## 3b. Регрессия на реальных данных — California Housing

**Датасет:** `sklearn.fetch_california_housing` — 20 640 домов, 8 признаков, цель — медианная цена.  
**Лосс:** MSE  
**Метрики:** RMSE + R² на тест-сете  
**Чекпоинт:** `checkpoints/reg_california.npz`

In [ ]:
# A2.
print("\n═══  A2. CALIFORNIA HOUSING  ═══")
import pathlib
from sklearn.datasets import fetch_california_housing

CKPT_DIR = pathlib.Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

ch = fetch_california_housing()
X_ch = ch.data.astype(np.float32)
y_ch = ch.target.astype(np.float32).reshape(-1, 1)

sc_x_ch, sc_y_ch = StandardScaler(), StandardScaler()
X_ch = sc_x_ch.fit_transform(X_ch).astype(np.float32)
y_ch = sc_y_ch.fit_transform(y_ch).astype(np.float32)

X_ch_tr, X_ch_tmp, y_ch_tr, y_ch_tmp = train_test_split(X_ch, y_ch, test_size=0.25, random_state=3)
X_ch_va, X_ch_te, y_ch_va, y_ch_te   = train_test_split(X_ch_tmp, y_ch_tmp, test_size=0.5, random_state=3)
print(f"  Train={X_ch_tr.shape[0]}, Val={X_ch_va.shape[0]}, Test={X_ch_te.shape[0]}")
# реальные данные интереснее синтетики — тут есть выбросы и нелинейности

def build_reg_ch(hs, Act=ReLU):
    m = Sequential(); d = X_ch.shape[1]
    for h in hs:
        m.add(Linear(d,h)).add(BatchNormalization(0.9)).add(ChannelwiseScaling(h))
        m.add(Act()).add(Dropout(0.2)); d = h
    return m.add(Linear(d, 1))

np.random.seed(42)
mdl_ch = build_reg_ch([128, 64])
tr_ch  = Trainer(mdl_ch, MSECriterion(), Adam(mdl_ch, lr=1e-3), rmse, cosine_lr,
                 checkpoint_path=CKPT_DIR/"reg_california.npz")
tr_ch.fit(X_ch_tr, y_ch_tr, X_ch_va, y_ch_va,
          epochs=60, bs=256, patience=10, warmup=3, lr=1e-3, verbose=True)

mdl_ch.evaluate()
pred_ch = mdl_ch.forward(X_ch_te)
r2_ch = r2_score(pred_ch, y_ch_te)
print(f"\n  California Housing — test RMSE={rmse(pred_ch, y_ch_te):.4f}  R²={r2_ch:.4f}")
print(f"  Чекпоинт сохранён: checkpoints/reg_california.npz")
# R² > 0.7 считается приемлемым для housing данных
if r2_ch > 0.7:
    print("  хороший результат для линейной модели на этом датасете")

tr_ch.plot("California Housing", "RMSE")


## 4. Многоклассовая классификация — CNN на MNIST (или load_digits 8×8 как fallback)

**Датасет:** `torchvision.datasets.MNIST` если доступен интернет, иначе `sklearn.load_digits` — аналогичная задача классификации рукописных цифр 0–9.  
**Архитектура:** Conv → ReLU → MaxPool → Conv → ReLU → MaxPool → Flatten → Linear → Dropout → Linear → LogSoftMax  
**Лосс:** ClassNLLCriterion (negative log-likelihood)  
**Метрика:** Accuracy


In [ ]:
# B.
print("\n═══  B. CLASSIFICATION  ═══")

def try_load_mnist():
    """Пытается загрузить MNIST через sklearn fetch_openml. Возвращает (X, y) или None."""
    try:
        from sklearn.datasets import fetch_openml
        print("  Загружаем MNIST (это может занять минуту)...")  # кешируется после первого раза
        mnist = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
        X = (mnist.data / 255.).astype(np.float32).reshape(-1, 1, 28, 28)
        y = mnist.target.astype(int)
        print(f"  MNIST загружен: {X.shape}")
        return X, y
    except Exception as e:
        print(f"  MNIST недоступен ({e}), используем load_digits.")
        return None

mnist_data = try_load_mnist()

if mnist_data is not None:
    # Настоящий MNIST 28×28
    Xd, yd = mnist_data
    # Для скорости берём 10k train + 2k val + 2k test
    # Для скорости: 10k train, оставляем 14k для train/val/test split
    idx = np.random.RandomState(42).permutation(len(yd))
    Xd, yd = Xd[idx[:14000]], yd[idx[:14000]]
    IMG_SIZE = 28
else:
    # Fallback: load_digits 8×8
    from sklearn.datasets import load_digits
    dig = load_digits()
    Xd = (dig.data / 16.).astype(np.float32).reshape(-1, 1, 8, 8)
    yd = dig.target
    IMG_SIZE = 8
    print(f"  load_digits: {Xd.shape}")

Xdt, Xdtmp, ydt, ydtmp = train_test_split(Xd, yd, test_size=0.25, random_state=42, stratify=yd)
Xdv, Xdte, ydv, ydte   = train_test_split(Xdtmp, ydtmp, test_size=0.5, random_state=42, stratify=ydtmp)

def oh(y, n=10): o=np.zeros((len(y),n),np.float32); o[np.arange(len(y)),y]=1.; return o
Ydt, Ydv, Ydte = oh(ydt), oh(ydv), oh(ydte)
def acc(p, t): return float((p.argmax(1)==t.argmax(1)).mean())
print(f"  Train={Xdt.shape[0]}, Val={Xdv.shape[0]}, Test={Xdte.shape[0]}, img={IMG_SIZE}×{IMG_SIZE}")

# два MaxPool(2,2) делят пространственные размеры на 4
# 8×8 → 2×2, 28×28 → 7×7
flat_dim = 32 * (IMG_SIZE // 4) * (IMG_SIZE // 4)
assert flat_dim > 0, f"flat_dim должен быть > 0, получили {flat_dim}"

np.random.seed(42)
cnn = Sequential()
cnn.add(Conv2d(1, 16, 3, padding=1)).add(ReLU())
cnn.add(MaxPool2d(2, 2))
cnn.add(Conv2d(16, 32, 3, padding=1)).add(ReLU())
cnn.add(MaxPool2d(2, 2))
cnn.add(Flatten(1, -1))                          # (N, flat_dim)
cnn.add(Linear(flat_dim, 64)).add(ReLU()).add(Dropout(0.3))
cnn.add(Linear(64, 10)).add(LogSoftMax())

tr_cnn = Trainer(cnn, ClassNLLCriterion(), Adam(cnn, lr=3e-4), acc, cosine_lr)
# bs=128 вместо 32 — в ~4× быстрее на MNIST при незначительной потере качества
tr_cnn.fit(Xdt, Ydt, Xdv, Ydv, epochs=80, bs=128, patience=15, warmup=3, lr=3e-4, verbose=True)
tr_cnn.plot("CNN-Digits", "Accuracy")

cnn.evaluate()
test_acc = acc(cnn.forward(Xdte), Ydte)
print(f"\n  Test Accuracy: {test_acc:.4f}")
if test_acc < 0.85:
    print("  !! accuracy ниже 85% — попробовать больше эпох или другой lr")


In [ ]:
# Кривые обучения CNN
tr_cnn.plot("CNN-Digits","Accuracy"); plt.show()
# Confusion matrix
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
cnn.evaluate()
y_pred = cnn.forward(Xdte).argmax(1)
cm = confusion_matrix(ydte, y_pred)
fig, ax = plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay(cm).plot(ax=ax, colorbar=False)
ax.set_title('CNN Digits — Confusion Matrix (test set)')
plt.tight_layout(); plt.show()

## 5. Свёрточный Автоэнкодер — Digits 8×8

**Архитектура:**  
- **Encoder:** Conv(1→16, s=2) → ELU → Dropout → Conv(16→32, s=2) → ELU → Flatten → Linear(128→64) → BN → ELU → Linear(64→**16**)  
- **Decoder:** Linear(16→64) → Linear(64→128) → Reshape(32,2,2) → Upsample×2 → Conv(32→16) → Upsample×2 → Conv(16→1) → Sigmoid  

**Лосс:** MSE4D (pixel-level reconstruction)  
**Метрика:** MSE (чем меньше, тем лучше)  
**Latent dim:** 16 (против 64 признаков входа — сжатие в 4×)

In [ ]:
# C. CONVOLUTIONAL AUTOENCODER
# Архитектура адаптируется под IMG_SIZE (8×8 или 28×28)
print("\n═══  C. CONVOLUTIONAL AUTOENCODER  ═══")

# flat_dim уже вычислен выше в секции классификации:
# IMG_SIZE=8  → 32*2*2 = 128
# IMG_SIZE=28 → 32*7*7 = 1568
# spatial_after_enc = IMG_SIZE // 4  (два stride=2 conv)
spatial = IMG_SIZE // 4        # 2 для 8×8, 7 для 28×28
ae_flat  = 32 * spatial * spatial

LATENT = 16  # latent dim — сжимаем в 16 чисел, можно поэкспериментировать с 8 или 32
np.random.seed(42)
ae = Sequential()
# encoder: stride=2 вместо MaxPool — сразу уменьшаем и считаем признаки
ae.add(Conv2d(1, 16, 3, stride=2, padding=1)).add(ELU()).add(Dropout(0.1))
ae.add(Conv2d(16, 32, 3, stride=2, padding=1)).add(ELU())
ae.add(Flatten(1, -1))
ae.add(Linear(ae_flat, 64)).add(BatchNormalization(0.9)).add(ChannelwiseScaling(64)).add(ELU())
ae.add(Linear(64, LATENT))  # бутылочное горлышко
# decoder: сначала FC разворачиваем до нужного размера, потом upsample + conv
ae.add(Linear(LATENT, 64)).add(ELU())
ae.add(Linear(64, ae_flat)).add(ReLU())
ae.add(Reshape(32, spatial, spatial))
ae.add(Upsample2d(2))  # nearest-neighbour, transposed conv был бы точнее но сложнее
ae.add(Conv2d(32, 16, 3, padding=1)).add(ELU())
ae.add(Upsample2d(2))
ae.add(Conv2d(16, 1, 3, padding=1)).add(Sigmoid())  # sigmoid: выход в [0,1] как пиксели

def mse4d(p, t): return float(np.mean((p - t) ** 2))

import pathlib
CKPT_DIR = pathlib.Path("checkpoints"); CKPT_DIR.mkdir(exist_ok=True)

opt_ae = Adam(ae, lr=1e-3, wd=1e-5)
tr_ae  = Trainer(ae, MSE4D(), opt_ae, mse4d, cosine_lr,
                 checkpoint_path=CKPT_DIR/"autoencoder.npz")
tr_ae.fit(Xdt, Xdt, Xdv, Xdv,
          epochs=80, bs=64, patience=15, warmup=3, lr=1e-3, verbose=True)
tr_ae.plot("Autoencoder", "MSE")

ae.evaluate()
pred_ae = ae.forward(Xdte)
mse_val = mse4d(pred_ae, Xdte)
psnr_db = 10.0 * np.log10(1.0 / max(mse_val, 1e-12))
print(f"\n  Test Reconstruction MSE: {mse_val:.6f}")
print(f"  Test PSNR: {psnr_db:.2f} dB")  # >20dB — приемлемо, >30dB — хорошо
print(f"  Чекпоинт: checkpoints/autoencoder.npz")

# Визуализация реконструкций
n = 8; orig = Xdte[:n]; rec = ae.forward(orig)
fig, ax = plt.subplots(2, n, figsize=(n*1.4, 2.8))
for i in range(n):
    ax[0,i].imshow(orig[i,0], cmap='gray_r', vmin=0, vmax=1); ax[0,i].axis('off')
    ax[1,i].imshow(rec[i,0],  cmap='gray_r', vmin=0, vmax=1); ax[1,i].axis('off')
ax[0,0].set_ylabel('Original', fontsize=8)
ax[1,0].set_ylabel('Recon',    fontsize=8)
plt.suptitle('Autoencoder: original vs reconstruction', fontsize=10)
plt.tight_layout(); plt.show()
print("\n✓ All done!")


In [ ]:
# Кривая обучения автоэнкодера
tr_ae.plot("Autoencoder","MSE")

# Реконструкции
ae.evaluate()
n = 10; orig = Xdte[:n]; rec = ae.forward(orig)
fig, axes = plt.subplots(2, n, figsize=(n*1.4, 2.8))
for i in range(n):
    axes[0,i].imshow(orig[i,0], cmap='gray_r', vmin=0, vmax=1); axes[0,i].axis('off')
    axes[1,i].imshow(rec[i,0],  cmap='gray_r', vmin=0, vmax=1); axes[1,i].axis('off')
axes[0,0].set_ylabel('Original', fontsize=8)
axes[1,0].set_ylabel('Recon',    fontsize=8)
plt.suptitle('Autoencoder: original vs reconstruction', fontsize=10)
plt.tight_layout(); plt.show()